In [1]:
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 46 — Choice entropy and the cost sensitivity of the LRT demand, by TAZ, station and segment (diagnostic)

Step 45 (`LRT_alternatives_demand.ipynb`, §6aq) gives every corridor TAZ pair two probabilities: the transit
share of the pair after the LRT, and the LRT's share of that transit, `P_L = 1 / (1 + exp(λ_T · dL))` with
`dL = GC_LRT − premium − GC_bus`. This step asks two things of those probabilities.

**1. How uncertain is the choice itself?** The binary entropy `H(p) = −p·log2 p − (1−p)·log2(1−p)` is 0 bits where
the choice is settled and 1 bit at a coin flip. It is computed per TAZ pair for the LRT against the bus / Metronit
and for transit against the car, and averaged with the trips as weights.

**2. Where does a minute of error in the LRT's cost move the demand most?** For a pair, one more generalized minute
on the LRT changes its LRT trips by

`dT_LRT / dc = −T_LRT · [ λ_T · (1 − P_L) + λ · (1 − S_new) · P_L ]`

The first term is riders going back to the bus or the Metronit; summed over the pairs it is `λ_T · Σ T_transit ·
P_L · (1 − P_L)`, the same `p(1 − p)` that the entropy grows with. The second term is the car shift given back
through the nest's logsum. The step does not re-derive the model: it reads the checked numpy copy of step 45's
pivot in `tools/build_scenario_workbook.py` (which stops if any LRT total is more than 0.5 trips from
`Output/alternatives/demand_summary.csv`) and takes the derivative by a central difference of ±0.05 generalized
minutes, pair by pair. The first term is also computed in closed form and compared.

The pair sensitivities are then summed three ways:

- **by TAZ** — the LRT trips lost if the LRT leg at this TAZ (as origin or as destination) costs one generalized
  minute more: half a minute of walking or waiting at weight 2, or one minute of riding;
- **by station** — the same for a minute at the station (boarding or alighting there);
- **by segment and direction** — the LRT trips lost if the running time of this segment is one minute longer.
  Every trip lost rode that segment, so the figure is also the change of the segment's own load.

All volumes are trips in the three hours of the period (06:00–09:00, 16:00–19:00), as in `demand_summary.csv`.
A diagnostic: it changes no number of the study.

In [2]:
import sys, time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from shapely.geometry import LineString

sys.path.insert(0, 'tools')
import build_scenario_workbook as M          # step 45's pivot in numpy on the committed skims of Output/scenario_tool/ (Metronit as today = M0)

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/alternatives/choice_sensitivity'; FIG = 'Output/figures'; os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)
LAM = M.DEFAULTS['lam']; LAM_T = LAM * M.DEFAULTS['ratio']      # 0.03 and 0.06, step 45's central values
H_FD = 0.05                                                      # half-width of the central difference, generalized minutes
REF = ('AM', 'BU_2050', 'main_ext', 'prioritized')               # the reference case of the study
ALT_LABEL = {'main_ext': 'Main route + extension', 'ext': 'Extension only'}; REG_LABEL = {'prioritized': 'Prioritized', 'unprioritized': 'Unprioritized'}

D = M.load(); TAZS = D['TAZS']; NT = len(TAZS)
chk, worst, ds = M.check_against_step45(D)
assert worst <= 0.5, f'the numpy pivot is {worst:.2f} LRT trips from step 45 — rebuild Output/scenario_tool/ (tools/scenario_metronit_skims.py)'
print(f'{NT} corridor TAZs; the pivot reproduces the 40 LRT totals of step 45 within {worst:.2f} trips; λ {LAM} / λ_T {LAM_T}')

174 corridor TAZs; the pivot reproduces the 40 LRT totals of step 45 within 0.05 trips; λ 0.03 / λ_T 0.06


## 1. Entropy and sensitivity per TAZ pair, for the 40 runs of step 45

In [3]:
def h2(p):
    """binary entropy in bits"""
    p = np.clip(p, 1e-12, 1 - 1e-12); return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
def shifted(per, sc, alt, reg, d):
    """the pivot with d generalized minutes added to the LRT cost of every pair (through the station-access input: 2 ends × d / 2)"""
    return M.pivot(D, per, sc, alt, reg, 'M0', {'st_access': M.DEFAULTS['st_access'] + d / 2})
def on_segments(X, a, b, nst):
    """load the pair matrix X on the line's segments between the pair's stations (1-based orders a, b; 0 = no LRT), by direction, and on the stations"""
    ok = (a > 0) & (X != 0); oa, ob, x = a[ok] - 1, b[ok] - 1, X[ok]; up = ob > oa
    d1 = np.zeros(nst); np.add.at(d1, oa[up], x[up]); np.add.at(d1, ob[up], -x[up])
    d2 = np.zeros(nst); np.add.at(d2, ob[~up], x[~up]); np.add.at(d2, oa[~up], -x[~up])
    board = np.zeros(nst); alight = np.zeros(nst); np.add.at(board, oa, x); np.add.at(alight, ob, x)
    return np.cumsum(d1)[:-1], np.cumsum(d2)[:-1], board, alight

RES = {}; rows = []; taz_rows = []; seg_rows = []; stn_rows = []; t0 = time.time()
for per in M.PERIODS:
    for sc in M.SCEN:
        for alt in M.ALTS:
            for reg in M.REGS:
                R0 = M.pivot(D, per, sc, alt, reg, 'M0'); Rp = shifted(per, sc, alt, reg, H_FD); Rm = shifted(per, sc, alt, reg, -H_FD)
                n = R0['car0'] + R0['tr0']; tr1, lrt, PL, a, b = R0['tr1'], R0['lrt'], R0['PL'], R0['a'], R0['b']; av = (a > 0) & (tr1 > 0)
                sens = (Rm['lrt'] - Rp['lrt']) / (2 * H_FD)             # = −dT_LRT/dc ≥ 0: LRT trips lost per +1 generalized minute on the LRT
                s_car = -PL * (Rp['tr1'] - Rm['tr1']) / (2 * H_FD)     # of which back to the car: the pair's transit trips shrink (zero where the pair has no car trips to give back)
                s_switch = sens - s_car                                 # of which back to the bus / Metronit
                closed = LAM_T * tr1 * PL * (1 - PL)                    # the same in closed form
                assert np.abs(closed - s_switch).max() < 1e-4 and sens.min() > -1e-9
                S1 = np.where(n > 0, tr1 / np.where(n > 0, n, 1), 0.0)       # the pair's transit share after the LRT
                HL = np.where(av, h2(PL), 0.0); HU = np.where(av, h2(S1), 0.0)
                st = D['ST'][alt]; nst = len(st); L1, L2, bo, al = on_segments(lrt, a, b, nst); S1s, S2s, sbo, sal = on_segments(sens, a, b, nst)
                one = {d: float(shifted(per, sc, alt, reg, d)['lrt'].sum()) for d in (-1.0, 1.0)}      # the full model at ∓1 minute, for the size of the curvature
                W = lrt.sum(); var = float((tr1 * PL * (1 - PL)).sum())
                rows.append({'period': per, 'scenario': sc, 'alternative': ALT_LABEL[alt], 'regime': REG_LABEL[reg], 'LRT trips': W, 'transit trips on the LRT pairs': float(tr1[av].sum()),
                             'entropy LRT vs bus/Metronit (bits, transit-trip mean)': float((tr1 * HL).sum() / tr1[av].sum()),
                             'entropy transit vs car (bits, all-trip mean)': float((n * HU).sum() / n[av].sum()),
                             'share of LRT trips, P_L < 0.3': float(lrt[PL < .3].sum() / W), 'share of LRT trips, P_L 0.3–0.7': float(lrt[(PL >= .3) & (PL <= .7)].sum() / W), 'share of LRT trips, P_L > 0.7': float(lrt[PL > .7].sum() / W),
                             'sum T·P_L·(1−P_L)': var, 'sampling sd of the LRT total (trips)': float(np.sqrt(var)),
                             'LRT trips lost per +1 gen. min': float(sens.sum()), 'of which back to bus/Metronit': float(s_switch.sum()), 'of which back to car': float(s_car.sum()),
                             '% of LRT trips per gen. min': float(100 * sens.sum() / W), 'full model: LRT at −1 min': one[-1.0], 'full model: LRT at +1 min': one[1.0],
                             'busiest segment: trips lost per +1 min of its running time': float(max(S1s.max(), S2s.max()))})
                key = (per, sc, alt, reg); tag = {'period': per, 'scenario': sc, 'alternative': alt, 'regime': reg}
                with np.errstate(divide='ignore', invalid='ignore'):
                    taz = pd.DataFrame({**tag, 'TAZ': TAZS, 'AggCode': D['ta'], 'lrt_origins': lrt.sum(1), 'lrt_destinations': lrt.sum(0), 'lost_per_min_origin_end': sens.sum(1), 'lost_per_min_destination_end': sens.sum(0)})
                    taz['lrt_ends'] = taz['lrt_origins'] + taz['lrt_destinations']; taz['lost_per_min'] = taz['lost_per_min_origin_end'] + taz['lost_per_min_destination_end']
                    taz['pct_per_min'] = np.where(taz['lrt_ends'] > 0, 100 * taz['lost_per_min'] / taz['lrt_ends'], np.nan)
                    wt = (tr1 * av).sum(1) + (tr1 * av).sum(0)
                    taz['entropy_lrt_vs_bus_bits'] = np.where(wt > 0, ((tr1 * HL).sum(1) + (tr1 * HL).sum(0)) / wt, np.nan); taz['lrt_share_of_transit'] = np.where(wt > 0, taz['lrt_ends'] / wt, np.nan)
                    seg = pd.DataFrame({**tag, 'from_station': st['station_id'].values[:-1], 'to_station': st['station_id'].values[1:], 'load_dir1_towards_Nazareth_end': L1, 'load_dir2_towards_TiratCarmel': L2,
                                        'lost_per_min_dir1': S1s, 'lost_per_min_dir2': S2s})
                    seg['pct_per_min_dir1'] = np.where(L1 > 0, 100 * S1s / L1, np.nan); seg['pct_per_min_dir2'] = np.where(L2 > 0, 100 * S2s / L2, np.nan)
                    stn = pd.DataFrame({**tag, 'station': st['station_id'].values, 'line': st['line'].values, 'boardings': bo, 'alightings': al, 'lost_per_min_boarding': sbo, 'lost_per_min_alighting': sal})
                    stn['pct_per_min'] = np.where(bo + al > 0, 100 * (sbo + sal) / (bo + al), np.nan)
                RES[key] = {'R': R0, 'sens': sens, 'taz': taz, 'seg': seg, 'stn': stn}; taz_rows.append(taz); seg_rows.append(seg); stn_rows.append(stn)
summary = pd.DataFrame(rows); summary.to_csv(f'{OUT}/choice_entropy_sensitivity_summary.csv', index=False, float_format='%.3f')
pd.concat(taz_rows).to_csv(f'{OUT}/sensitivity_by_taz.csv.gz', index=False, float_format='%.3f', compression='gzip')
pd.concat(seg_rows).to_csv(f'{OUT}/sensitivity_by_segment.csv', index=False, float_format='%.2f')
pd.concat(stn_rows).to_csv(f'{OUT}/sensitivity_by_station.csv', index=False, float_format='%.2f')
print(f'{len(RES)} runs in {time.time() - t0:.0f} s')
show = summary[summary['alternative'] == 'Main route + extension'][['period', 'scenario', 'regime', 'LRT trips', 'entropy LRT vs bus/Metronit (bits, transit-trip mean)', 'entropy transit vs car (bits, all-trip mean)', 'share of LRT trips, P_L 0.3–0.7',
                                                                    'sampling sd of the LRT total (trips)', 'LRT trips lost per +1 gen. min', 'of which back to car', '% of LRT trips per gen. min']].copy()
show.columns = ['period', 'scenario', 'regime', 'LRT', 'H LRT|transit', 'H transit|all', 'LRT at P_L 0.3–0.7', 'sampling sd', 'lost per min', 'to car', '% per min']
print(show.round({'LRT': 0, 'H LRT|transit': 3, 'H transit|all': 3, 'LRT at P_L 0.3–0.7': 3, 'sampling sd': 1, 'lost per min': 1, 'to car': 1, '% per min': 2}).to_string(index=False))

40 runs in 8 s
period scenario        regime    LRT  H LRT|transit  H transit|all  LRT at P_L 0.3–0.7  sampling sd  lost per min  to car  % per min
    AM     2022   Prioritized 5193.0          0.875          0.254               0.706         48.7         155.2    12.8       2.99
    AM     2022 Unprioritized 4209.0          0.827          0.247               0.627         46.8         140.7     9.4       3.34
    AM  BU_2040   Prioritized 6309.0          0.879          0.248               0.715         54.2         191.7    15.5       3.04
    AM  BU_2040 Unprioritized 4999.0          0.825          0.241               0.634         51.8         171.4    10.7       3.43
    AM  BU_2050   Prioritized 7433.0          0.879          0.328               0.701         58.2         236.0    33.1       3.18
    AM  BU_2050 Unprioritized 5700.0          0.829          0.319               0.651         55.3         203.3    20.0       3.57
    AM  HS_2040   Prioritized 7476.0          0.868   

## 2. How far the per-minute figure holds: the full model against the straight line

The derivative is local. The cell below shifts the LRT cost of every pair by −10 … +10 generalized minutes, reruns
the pivot, and sets the result beside the straight line through the central case. It also places the two regimes
of step 45 on that curve: the Unprioritized run is the Prioritized one with longer running times, pair by pair.

In [4]:
SHIFTS = np.arange(-10, 10.01, 1.0); curve_rows = []
for reg in M.REGS:
    per, sc, alt, _ = REF; base = summary[(summary['period'] == per) & (summary['scenario'] == sc) & (summary['alternative'] == ALT_LABEL[alt]) & (summary['regime'] == REG_LABEL[reg])].iloc[0]
    for d in SHIFTS:
        curve_rows.append({'period': per, 'scenario': sc, 'alternative': alt, 'regime': reg, 'shift_gen_min': d, 'LRT full model': float(shifted(per, sc, alt, reg, d)['lrt'].sum()),
                           'LRT straight line': base['LRT trips'] - base['LRT trips lost per +1 gen. min'] * d})
curve = pd.DataFrame(curve_rows); curve['line minus model'] = curve['LRT straight line'] - curve['LRT full model']; curve.to_csv(f'{OUT}/lrt_response_to_cost_shift.csv', index=False, float_format='%.1f')
# the regime gap of step 45 predicted from the Prioritized run's pair sensitivities and the pair-by-pair difference of the two LRT costs
per, sc, alt, _ = REF; gP = np.nan_to_num(D['GCL'][(per, 'M0', alt, 'prioritized')]); gU = np.nan_to_num(D['GCL'][(per, 'M0', alt, 'unprioritized')]); RP = RES[(per, sc, alt, 'prioritized')]; RU = RES[(per, sc, alt, 'unprioritized')]
both = (RP['R']['a'] > 0) & (RU['R']['a'] > 0); dgc = np.where(both, gU - gP, 0.0); wP = RP['R']['lrt']
gap_true = RP['R']['lrt'].sum() - RU['R']['lrt'].sum(); gap_lin = float((RP['sens'] * dgc).sum()); gap_mid = float(((RP['sens'] + RU['sens']) / 2 * dgc).sum())
print(f"{per} {sc}, {ALT_LABEL[alt]}: Unprioritized LRT cost is {float((wP * dgc).sum() / wP.sum()):.1f} generalized minutes above Prioritized (LRT-trip mean); "
      f"step 45's gap {gap_true:,.0f} trips; from the Prioritized sensitivities {gap_lin:,.0f}; from the mean of the two runs' sensitivities {gap_mid:,.0f}")
print(curve[curve['shift_gen_min'].isin([-10, -5, -2, -1, 1, 2, 5, 10])].pivot(index='shift_gen_min', columns='regime', values=['LRT full model', 'LRT straight line']).round(0).to_string())

AM BU_2050, Main route + extension: Unprioritized LRT cost is 8.3 generalized minutes above Prioritized (LRT-trip mean); step 45's gap 1,733 trips; from the Prioritized sensitivities 1,969; from the mean of the two runs' sensitivities 1,723
              LRT full model               LRT straight line              
regime           prioritized unprioritized       prioritized unprioritized
shift_gen_min                                                             
-10.0                 9925.0        7924.0            9793.0        7733.0
-5.0                  8650.0        6767.0            8613.0        6717.0
-2.0                  7911.0        6115.0            7905.0        6106.0
-1.0                  7670.0        5905.0            7669.0        5903.0
 1.0                  7198.0        5499.0            7197.0        5496.0
 2.0                  6967.0        5302.0            6961.0        5293.0
 5.0                  6297.0        4738.0            6253.0        4683.0
 10.0    

## 3. Figures: the distribution of the choice, and the sensitivity by TAZ and by segment (reference case AM 2050 BU, through line)

In [5]:
per, sc, alt, _ = REF; bins = np.linspace(0, 1, 11); mid = (bins[:-1] + bins[1:]) / 2
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))
ax = axes[0]
for k, (reg, c) in enumerate((('prioritized', BLUE), ('unprioritized', ORANGE))):
    R = RES[(per, sc, alt, reg)]['R']; ok = R['lrt'] > 0; hgt, _ = np.histogram(R['PL'][ok], bins=bins, weights=R['lrt'][ok])
    ax.bar(mid + (k - .5) * .04, hgt, width=.038, color=c, label=f"{REG_LABEL[reg]} ({R['lrt'].sum():,.0f} LRT trips)")
ax.set_xticks(bins); ax.set_xlabel("the pair's LRT share of transit, P_L"); ax.set_ylabel('LRT trips, AM three hours'); ax.set_title('Where the LRT trips sit on the choice: most are near an even split', fontsize=10, loc='left'); ax.legend(frameon=False, fontsize=8)
ax = axes[1]; x = np.linspace(-40, 40, 400); ax.plot(x, 1 / (1 + np.exp(LAM_T * x)), color=INK, lw=1.4, label=f'P_L at λ_T = {LAM_T:g}')
R = RES[(per, sc, alt, 'prioritized')]['R']; ok = R['lrt'] > 0; dl = np.log(1 / R['PL'][ok] - 1) / LAM_T; ax2 = ax.twinx(); ax2.hist(dl, bins=np.arange(-40, 41, 4), weights=R['tr1'][ok], color=AQUA, alpha=.35); ax2.set_ylabel('transit trips on the LRT pairs (bars)', fontsize=8); ax2.tick_params(labelsize=7)
ax.axvline(0, color=AXIS, lw=.8); ax.set_xlabel('dL = LRT cost − premium − best bus / Metronit cost, generalized minutes'); ax.set_ylabel('P_L'); ax.set_title('The split is flat: 10 minutes of advantage give 65 / 35', fontsize=10, loc='left'); ax.legend(frameon=False, fontsize=8, loc='upper right')
ax = axes[2]
for reg, c in (('prioritized', BLUE), ('unprioritized', ORANGE)):
    cv = curve[curve['regime'] == reg]; ax.plot(cv['shift_gen_min'], cv['LRT full model'], color=c, lw=1.8, label=f'{REG_LABEL[reg]}: full model'); ax.plot(cv['shift_gen_min'], cv['LRT straight line'], color=c, lw=1, ls='--', label=f'{REG_LABEL[reg]}: straight line from the derivative')
ax.axvline(0, color=AXIS, lw=.8); ax.set_xlabel('generalized minutes added to the LRT cost of every pair'); ax.set_ylabel('LRT trips, AM three hours'); ax.set_title('The per-minute figure holds within a few minutes', fontsize=10, loc='left'); ax.legend(frameon=False, fontsize=8)
for ax in axes: ax.grid(color=GRID, lw=.5); ax.set_axisbelow(True); [ax.spines[s].set_visible(False) for s in ('top', 'right')]
fig.suptitle(f'Choice between the LRT and the bus / Metronit — {per} {sc}, {ALT_LABEL[alt]}', fontsize=11); plt.tight_layout(); plt.savefig(f'{FIG}/lrt_choice_entropy_distribution.png', dpi=130); plt.close()

tzg = gpd.read_file('Input/TAZ_North/TAZ_North.shp', engine='pyogrio'); tzg['TAZ'] = tzg['TAZ_NUMBER'].astype(int); tzg = tzg.drop_duplicates('TAZ').set_index('TAZ'); corr = tzg.loc[[t for t in TAZS if t in tzg.index]]
STN = D['ST'][alt]; line_xy = STN[['x', 'y']].values; XMIN, XMAX, YMIN, YMAX = 188000, 236000, 727000, 752000
def base(ax, title):
    tzg.boundary.plot(ax=ax, color='#eeede8', lw=.25); corr.boundary.plot(ax=ax, color=AXIS, lw=.35); ax.set_xlim(XMIN, XMAX); ax.set_ylim(YMIN, YMAX); ax.set_axis_off(); ax.set_title(title, fontsize=9, loc='left')
def choropleth(ax, series, title, cmap, norm, label):
    base(ax, title); g = corr.assign(v=series.reindex(corr.index).values); g.plot(ax=ax, column='v', cmap=cmap, norm=norm, edgecolor=AXIS, linewidth=.3, missing_kwds={'color': '#f4f3ef'})
    ax.plot(line_xy[:, 0], line_xy[:, 1], color=INK, lw=1.1, alpha=.8); ax.scatter(line_xy[:, 0], line_xy[:, 1], s=5, color=INK, zorder=5)
    cb = plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=.55, pad=.01); cb.set_label(label, fontsize=8); cb.ax.tick_params(labelsize=7)
for reg in M.REGS:
    T = RES[(per, sc, alt, reg)]['taz'].set_index('TAZ'); small = T['lrt_ends'] < 1.0       # TAZs with under one LRT trip end: shares and entropies not drawn
    fig, axes = plt.subplots(2, 2, figsize=(19, 10.5))
    choropleth(axes[0, 0], T['lrt_ends'].where(T['lrt_ends'] > 0), 'LRT trip ends (origins + destinations)', 'YlGnBu', Normalize(0, T['lrt_ends'].quantile(.98)), 'LRT trip ends per TAZ, three hours')
    choropleth(axes[0, 1], T['lost_per_min'].where(T['lrt_ends'] > 0), 'LRT trips lost per generalized minute added to the LRT leg at the TAZ', 'OrRd', Normalize(0, T['lost_per_min'].quantile(.98)), 'trips per generalized minute, three hours')
    choropleth(axes[1, 0], T['pct_per_min'].mask(small), "The same as a share of the TAZ's LRT trips", 'OrRd', Normalize(1.5, 5.0), '% of the LRT trips per generalized minute')
    choropleth(axes[1, 1], T['entropy_lrt_vs_bus_bits'].mask(small), 'Entropy of the choice LRT against bus / Metronit (1 bit = an even split)', 'PuBuGn', Normalize(0.5, 1.0), 'bits, transit-trip mean over the pairs of the TAZ')
    fig.suptitle(f'Where a minute of LRT cost moves the demand — {per} {sc}, {ALT_LABEL[alt]}, {REG_LABEL[reg]} (one generalized minute = half a minute of walking or waiting, or a minute of riding)', fontsize=11)
    plt.tight_layout(); plt.savefig(f'{FIG}/lrt_choice_sensitivity_map_{per}_{sc}_{reg}.png', dpi=130); plt.close()

def offset(seg_, d):
    try: return seg_.parallel_offset(abs(d), 'left' if d > 0 else 'right')
    except Exception: return seg_
segP = RES[(per, sc, alt, 'prioritized')]['seg']; segU = RES[(per, sc, alt, 'unprioritized')]['seg']; wmax = segP[['lost_per_min_dir1', 'lost_per_min_dir2']].values.max()
fig = plt.figure(figsize=(19, 10)); gs = fig.add_gridspec(2, 2, width_ratios=[1.25, 1]); axm = fig.add_subplot(gs[:, 0]); base(axm, f'LRT trips lost per minute added to the running time of the segment (width, max {wmax:,.0f} per minute) — Prioritized')
for k, s in segP.iterrows():
    g = LineString([line_xy[k], line_xy[k + 1]])
    for col, c, sign in (('lost_per_min_dir1', BLUE, 1), ('lost_per_min_dir2', ORANGE, -1)):
        w = s[col] / wmax * 10
        if w > .05:
            o = offset(g, sign * 130); xs, ys = (o.xy if o.geom_type == 'LineString' else list(o.geoms)[0].xy); axm.plot(xs, ys, color=c, lw=max(w, .4), solid_capstyle='butt', alpha=.9)
axm.scatter(line_xy[:, 0], line_xy[:, 1], s=9, facecolor='white', edgecolor=INK, lw=.6, zorder=5)
for k in list(range(0, len(STN), 4)) + [len(STN) - 1]: axm.annotate(STN['station_id'].iloc[k], line_xy[k], xytext=(4, 4), textcoords='offset points', fontsize=6.5, color=INK2)
axm.plot([], [], color=BLUE, lw=4, label='towards Nazareth'); axm.plot([], [], color=ORANGE, lw=4, label='towards Tirat Carmel'); axm.legend(loc='lower right', fontsize=8, frameon=False)
xs_ = np.arange(len(segP)); labs = [f"{f}–{t}" for f, t in zip(segP['from_station'], segP['to_station'])]
ax = fig.add_subplot(gs[0, 1]); ax.bar(xs_ - .2, segP['load_dir1_towards_Nazareth_end'], width=.4, color=BLUE, label='towards Nazareth'); ax.bar(xs_ + .2, segP['load_dir2_towards_TiratCarmel'], width=.4, color=ORANGE, label='towards Tirat Carmel')
ax.set_ylabel('LRT trips on the segment, three hours'); ax.set_title('Segment load (Prioritized)', fontsize=9, loc='left'); ax.legend(frameon=False, fontsize=8); ax.set_xticks(xs_[::2]); ax.set_xticklabels([])
ax_ = fig.add_subplot(gs[1, 1]); ax_.bar(xs_ - .2, segP['lost_per_min_dir1'], width=.4, color=BLUE); ax_.bar(xs_ + .2, segP['lost_per_min_dir2'], width=.4, color=ORANGE)
ax_.plot(xs_ - .2, segU['lost_per_min_dir1'], color=INK, lw=0, marker='_', ms=5, label='Unprioritized'); ax_.plot(xs_ + .2, segU['lost_per_min_dir2'], color=INK, lw=0, marker='_', ms=5)
ax_.set_ylabel('LRT trips lost per minute added to the segment'); ax_.set_title('Sensitivity of the segment (bars Prioritized, marks Unprioritized)', fontsize=9, loc='left'); ax_.legend(frameon=False, fontsize=8); ax_.set_xticks(xs_[::2]); ax_.set_xticklabels(labs[::2], rotation=90, fontsize=6.5)
for a_ in (ax, ax_): a_.grid(axis='y', color=GRID, lw=.5); a_.set_axisbelow(True); [a_.spines[s].set_visible(False) for s in ('top', 'right')]
fig.suptitle(f'Sensitivity of the LRT demand to the running time, by segment and direction — {per} {sc}, {ALT_LABEL[alt]}', fontsize=11); plt.tight_layout(); plt.savefig(f'{FIG}/lrt_choice_sensitivity_segments_{per}_{sc}.png', dpi=130); plt.close()

r = summary[(summary['period'] == per) & (summary['scenario'] == sc) & (summary['alternative'] == ALT_LABEL[alt]) & (summary['regime'] == 'Prioritized')].iloc[0]
top = segP.assign(m=segP[['lost_per_min_dir1', 'lost_per_min_dir2']].max(axis=1)).nlargest(5, 'm')
print(f"{per} {sc}, {ALT_LABEL[alt]}, Prioritized: {r['LRT trips']:,.0f} LRT trips; lost per +1 generalized minute on every pair {r['LRT trips lost per +1 gen. min']:,.0f} ({r['% of LRT trips per gen. min']:.2f} %), "
      f"of which {r['of which back to car']:,.0f} to the car; sampling sd {r['sampling sd of the LRT total (trips)']:,.0f}")
print('the five most sensitive segments (trips lost per minute of running time, towards Nazareth / towards Tirat Carmel; % of the load):')
print(top[['from_station', 'to_station', 'load_dir1_towards_Nazareth_end', 'load_dir2_towards_TiratCarmel', 'lost_per_min_dir1', 'lost_per_min_dir2', 'pct_per_min_dir1', 'pct_per_min_dir2']].round(1).to_string(index=False))
T = RES[REF]['taz']; ar = T.groupby('AggCode')[['lrt_ends', 'lost_per_min']].sum(); ar['pct_per_min'] = 100 * ar['lost_per_min'] / ar['lrt_ends']; ar.index = [f"{c} {D['ANAME'].get(c, '')}" for c in ar.index]
print('by corridor area, the ten largest (LRT trip ends; trips lost per generalized minute at the area; %):'); print(ar.nlargest(10, 'lost_per_min').round(1).to_string())
st_ = RES[REF]['stn'].assign(lost=lambda d_: d_['lost_per_min_boarding'] + d_['lost_per_min_alighting']).nlargest(6, 'lost')
print('the six most sensitive stations (boardings, alightings, trips lost per generalized minute at the station, %):'); print(st_[['station', 'line', 'boardings', 'alightings', 'lost', 'pct_per_min']].round(1).to_string(index=False))

AM BU_2050, Main route + extension, Prioritized: 7,433 LRT trips; lost per +1 generalized minute on every pair 236 (3.18 %), of which 33 to the car; sampling sd 58
the five most sensitive segments (trips lost per minute of running time, towards Nazareth / towards Tirat Carmel; % of the load):
from_station to_station  load_dir1_towards_Nazareth_end  load_dir2_towards_TiratCarmel  lost_per_min_dir1  lost_per_min_dir2  pct_per_min_dir1  pct_per_min_dir2
         S24        M02                           504.3                         2344.7               14.4               75.6               2.8               3.2
         S23        S24                           671.8                         1889.9               20.4               60.3               3.0               3.2
         S22        S23                           681.2                         1872.4               20.7               59.7               3.0               3.2
         S03        S04                          1840.0       

## 4. Results

**The choice is close to an even split; the car side is settled.** In the reference case (AM 2050 BU, main route +
extension, Prioritized) the entropy of the LRT against the bus / Metronit is **0.88 bits** of a possible 1
(0.82–0.88 over the 20 through-line runs; 0.59–0.65 for the extension alone, where the LRT is the weaker option on
more pairs). 70 % of the 7,433 LRT trips come from TAZ pairs where the LRT takes 30–70 % of the transit trips and
22 % from pairs where it takes more than 70 %. Transit against the car is at 0.33 bits (0.23–0.35 over the runs).
The entropy is high partly by construction: at λ_T = 0.06 an advantage of 10 generalized minutes gives a 65 / 35
split.

**It is not the uncertainty of the forecast.** If every traveller drew a mode with these probabilities the LRT total
would have a standard deviation of 58 trips (0.8 %). The planning range of the study (5,532–10,537) comes from the
parameters, which the entropy does not see.

**One generalized minute on the LRT is worth 236 trips, 3.2 % of the line** (reference case): 203 go back to the bus
or the Metronit (λ_T · Σ T · P_L · (1 − P_L); the closed form agrees with the difference to 1e-4 trips per pair) and
33 to the car. Over the through-line runs the figure is 3.0–3.6 % per minute (Unprioritized 3.3–3.6, Prioritized
3.0–3.2); for the extension alone up to 4.1 %. The straight line stays within about 1 % of the full model up to ± 5
minutes and is 1–6 % off at ± 10. It reproduces step 45's regime gap: the Unprioritized LRT is 8.3 generalized
minutes dearer (LRT-trip mean), the gap is 1,733 trips, and the mean of the two runs' pair sensitivities × the
pair's cost difference gives 1,723 (the Prioritized run's sensitivities alone: 1,969).

**On the extension the sensitivity follows the load; the main route holds its riders more firmly.** An extension
segment loses 3.0–3.4 % of its load per minute of running time (S01–S02: 4 %), so the segments that cost most are the loaded
ones: S24–M02 towards Tirat Carmel (76 trips per minute), S19–S24 in the same direction (57–60), S03–S06 and
S12–S15 towards Nazareth (57–59). East of M03 a segment loses 0.9–2.3 % of its load per minute and under 25 trips. By
area, a generalized minute at the LRT leg costs most in Tirat Carmel (80 trips), Bat Galim – Kiryat Eliezer (64),
Matam – Neot Peres (42), Hamifrats (37) and the lower city (37). As a share of the area's LRT trips it is 2.7–3.2 %
along the Haifa trunk, 3.6–4.1 % where the LRT takes a smaller share of transit (the Krayot by feeder, Matam, Ein
Hayam: LRT share 0.32–0.40) and 1.2 % in Nazareth (LRT share 0.75, entropy 0.62 bits).

**What it means for the work.** No single zone or segment carries the risk on the extension: the LRT total is
exposed to the level of the LRT's cost against the bus everywhere at once — the walk and wait weights, the
transfer penalties, the running regime, the premium. Without the premium (5 minutes over the bus, 2.5 over the
Metronit) the reference case is 6,622 trips, 11 % lower; at 2.5 / 1.5 it is 7,055. λ_T itself matters less: 7,942
at 0.04 and 6,942 at 0.10 (± 7 %), because pairs above and below an even split move in opposite directions.

## 5. Outputs and limits

**Outputs** (`Output/alternatives/choice_sensitivity/`). `choice_entropy_sensitivity_summary.csv` (the 40 runs),
`sensitivity_by_taz.csv.gz`, `sensitivity_by_station.csv`, `sensitivity_by_segment.csv` (every run, long format),
`lrt_response_to_cost_shift.csv`; figures `Output/figures/lrt_choice_entropy_distribution.png`,
`lrt_choice_sensitivity_map_AM_BU_2050_{prioritized,unprioritized}.png`, `lrt_choice_sensitivity_segments_AM_BU_2050.png`.

**Limits.** A local derivative of step 45's model: it carries every limit of that model and says nothing about
whether λ_T, the weights or the premium are right. The boarding and alighting stations are held fixed. A minute on
a segment is charged to the trips that ride it now. The entropy depends on λ_T: a larger λ_T lowers it without any
new information. The Metronit as today (the central case) only.